# Людина-в-циклі: Ворота перед дією, розподіл за рівнями ризику та аудит логів

README для цього уроку вводить поняття Людина-в-циклі коротким фрагментом, який запитує користувача `APPROVE` або `REJECT` після того, як агент вже сформував відповідь. Цей підхід є хорошою відправною точкою, але виробничі імплементації HITL зазвичай потребують ще трьох додаткових компонентів:

1. **Ворота перед дією**, які запускаються **перед тим**, як агент виконає ризиковий крок, щоб тримати під контролем витрати, необоротність і затримку.
2. **Розподіл за рівнями ризику**, щоб дії з низьким ризиком виконувалися автоматично, дії з середнім ризиком затверджувалися пакетно, а дії з високим ризиком блокувалися людиною.
3. **Аудиторський журнал плюс цикл ревізії**, щоб кожне рішення воріт записувалося у форматі JSONL, а відхилення спричиняло повторний запит агента зі структурованою причиною замість просто виведення `Revising...`.

Цей ноутбук будує кожен з цих елементів на основі тих же примітивів, що й `06-system-message-framework.ipynb`. Він працює наскрізь у режимі `DEMO_MODE = True` (без інтерактивного вводу) або з реальними підказками `input()`, коли `DEMO_MODE = False`. Примітка: у DEMO_MODE повторна спроба третьої цілі реалізована скриптово, щоб було видно механіку циклу наскрізь. Реальна переатестація з ревізією вимагає `DEMO_MODE = False` і оператора.

**Вийшло за межі цього уроку (розглядається в інших уроках):** автентифікація та контроль доступу (Урок 06 README загроза 2), проміжне програмне забезпечення виклику інструментів (Урок 14 глибокий аналіз MAF), шаблони дебатів з кількома агентами.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Шаблон 1: Ворота перед дією

Фрагмент HITL у README спочатку викликає агента, а потім просить користувача схвалити результат. Це є **поток після дії**. Агент вже виконався, тож вартість виклику LLM вже сплачена, і будь-який побічний ефект (надісланий електронний лист, запис у базі даних, опублікований коментар) уже стався.

**Потік перед дією** вставляє ворота перед тим, як агент виконає ризиковий крок. Агент пропонує дію, ворота вирішують, виконувати її чи ні, і побічний ефект відбувається лише за дозволом.

| Аспект | Схвалення після дії (фрагмент README) | Ворота перед дією (ця записна книжка) |
|---|---|---|
| Коли виконується схвалення? | Після того, як агент видав результат | Перед будь-яким виконанням побічного ефекту |
| Вартість LLM при відмові | Вже сплачена | Сплачується лише за пропозицію, а не за дію |
| Незворотні побічні ефекти | Можливі (дія вже відбулася) | Запобігається |
| Чіткість аудиту | Схвалення — це оператор виведення | Схвалення — це JSON-запис з часовою міткою, дією, причиною |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Шаблон 2: Розподіл за рівнями ризику

Не кожна дія потребує людського затвердження. Тільки читання через публічний API — це одна справа, а надсилання електронного листа клієнту — зовсім інша. Однакове ставлення до них марнує увагу оператора і уповільнює агента.

Просту модель з 3 рівнів:

| Рівень | Приклади | Процедура затвердження |
|---|---|---|
| `низький` (тільки читання) | Пошук у базі знань, пошук варіантів рейсів, отримання публічної веб-сторінки | Автовиконання, запис для аудиту |
| `середній` (недорога зміна) | Кешування результату, збільшення лічильника, планування нагадування | Автовиконання, але з щоденним пакетом для огляду |
| `високий` (зовнішній або незворотній) | Надсилання електронного листа, списання коштів, публікація в публічному каналі | Блокування до людського затвердження |

Це один із способів розподілу за рівнями. Виробничі системи часто використовують більш детальні рівні (наприклад, рівні дозволів AWS IAM, розподіл доступу за ролями). Версія з 3 рівнів нижче — найменша корисна версія для агента, що поєднує операції лише для читання і з побічним ефектом.

Класифікатор нижче використовує евристику ключових слів, щоб демонстрація залишалася детермінованою й недорогою. У виробничій системі ви замінили б його на навчений класифікатор або движок політик.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Патерн 3: Аудит журналу та цикл ревізій

`print("Response approved.")` — це не аудит журнал. Для довіри кожне рішення воріт має бути зафіксоване як структурована подія, яку ви зможете пізніше запитувати, відтворювати або додавати до огляду інцидентів.

Дві складові:

1. **Читання лише у кінець (append-only) JSONL.** Один рядок на рішення, з відміткою часу, дією, рівнем, рішенням, причиною. Легко шукати через grep, легко надсилати до справжнього сховища журналів пізніше.
2. **Цикл ревізії при відхиленні.** Коли ворота повертають `deny`, агент повторно підказує сам себе з аргументом причини відхилення у контексті, щоб наступна пропозиція могла уникнути проблеми.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Додаткові ресурси

Кілька інших публічних проєктів реалізують варіації цих шаблонів HITL. Порівняйте підходи, щоб знайти той, що підходить вашому стеку:

- **LangChain** оболонки інструментів з людиною в циклі ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): оболонки інструментів, які призупиняють виконання для введення людиною.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ цю структуру було змінено): використовує роль агента спеціально для представлення людини в багатоеагентних розмовах.
- **Microsoft Agent Framework (MAF)** проміжне програмне забезпечення для виклику функцій ([docs](https://learn.microsoft.com/agent-framework/)): проміжне ПЗ, що виконується навколо кожного виклику інструменту/функції, підходить для встановлення логіки контролю та процесів затвердження.

Кожен проєкт по-різному обробляє три підшаблони: LangChain обгортає їх як інструменти, AutoGen використовує роль агента, а Microsoft Agent Framework застосовує проміжне ПЗ для виклику функцій. Ознайомтесь з однією або двома реалізаціями повністю, перш ніж обирати дизайн для свого агента.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
